In [1]:
# open mat file with h5py
file_path = "/home/ppg/eyetracking/GFM-for-eyetracker/data/raw/eSEEd_v2/data_v2.mat"
import h5py
file = h5py.File(file_path, 'r')
print("Keys in the .mat file:")
for key in file.keys():
    print(key)


Keys in the .mat file:
#refs#
#subsystem#
Data


In [2]:
# ==============================================================================
# SUMMARY: How to Access Gaze Data from eSEEd_v2 .mat file
# ==============================================================================

def extract_gaze_data(h5_file, subject_idx=0, video_idx=0):
    """
    Extract gaze data from eSEEd_v2 dataset.
    
    Args:
        h5_file: Open h5py.File object
        subject_idx: Index of subject in the video dataset (0-47)
        video_idx: Always 0 (dataset has shape (48, 1))
    
    Returns:
        tuple: (gaze_items_list, video_group)
            - gaze_items_list: List of gaze data arrays
            - video_group: h5py Group containing 'gaze', 'pupil', 'blinks', 'annotation'
    """
    # Navigate to Data/video
    data_group = h5_file["Data"]
    video_dataset = data_group['video']
    
    # Get reference to specific subject's video
    video_ref = video_dataset[subject_idx, video_idx]
    video_group = h5_file[video_ref]
    
    # Extract gaze data (stored as object references)
    gaze_refs_dataset = video_group['gaze']
    gaze_items = []
    
    for i in range(len(gaze_refs_dataset)):
        gaze_ref = gaze_refs_dataset[i, 0]
        gaze_data = h5_file[gaze_ref][:]
        gaze_items.append(gaze_data)
    
    return gaze_items, video_group


# Example usage
print("="*80)
print("ACCESSING GAZE DATA")
print("="*80)

# Get gaze data for subject 0
gaze_data, video = extract_gaze_data(file, subject_idx=0, video_idx=0)

print(f"\nSubject 0, Video 0:")
print(f"  Number of gaze items: {len(gaze_data)}")
print(f"  Each gaze item shape: {gaze_data[0].shape}")
print(f"\nFirst 5 gaze items:")
for i in range(min(5, len(gaze_data))):
    print(f"  {i}: {gaze_data[i][0]}")

print(f"\nOther data available in video:")
for key in video.keys():
    ref_dataset = video[key]
    print(f"  - {key}: {len(ref_dataset)} items")

# Access pupil data similarly
print(f"\nAccessing pupil data:")
pupil_refs = video['pupil']
pupil_item_0 = file[pupil_refs[0, 0]][:]
print(f"  Pupil item 0: {pupil_item_0}")

# Access annotation
print(f"\nAccessing annotation:")
annotation_refs = video['annotation']
print(f"  Annotation shape: {annotation_refs.shape}")
if len(annotation_refs) > 0:
    try:
        annot = file[annotation_refs[0, 0]][:]
        print(f"  Annotation data: {annot}")
    except:
        print(f"  Annotation format may need special handling")

print("\n" + "="*80)
print("Note: The gaze data structure in this file appears to differ from")
print("the specification. Each gaze 'item' is a (1,6) array rather than")
print("the expected (N, 21) format with all gaze parameters.")
print("="*80)

ACCESSING GAZE DATA

Subject 0, Video 0:
  Number of gaze items: 10
  Each gaze item shape: (1, 6)

First 5 gaze items:
  0: [3707764736          2          1          1          1          1]
  1: [3707764736          2          1          1          2          1]
  2: [3707764736          2          1          1          3          1]
  3: [3707764736          2          1          1          4          1]
  4: [3707764736          2          1          1          5          1]

Other data available in video:
  - annotation: 10 items
  - blinks: 10 items
  - gaze: 10 items
  - pupil: 10 items

Accessing pupil data:
  Pupil item 0: [[3707764736          2          1          1         11          1]]

Accessing annotation:
  Annotation shape: (10, 1)
  Annotation data: [[3707764736          2          1          1         31          1]]

Note: The gaze data structure in this file appears to differ from
the specification. Each gaze 'item' is a (1,6) array rather than
the expected (N, 

In [13]:
h5_file = file
data_group = h5_file["Data"]
video_dataset = data_group['video']

# Get reference to specific subject's video
video_ref = video_dataset[0, 0]
video_group = h5_file[video_ref]

# Extract gaze data (stored as object references)
gaze_refs_dataset = video_group['gaze']
gaze_items = []

for i in range(len(gaze_refs_dataset)):
    gaze_ref = gaze_refs_dataset[i, 0]
    gaze_data = h5_file[gaze_ref][:]
    gaze_items.append(gaze_data)

In [8]:
gaze_ref = gaze_refs_dataset[0, 0]
h5_file[gaze_ref][0]

array([3707764736,          2,          1,          1,          1,
                1], dtype=uint32)

=== Exploring Gaze Data Structure ===

Gaze dataset attributes:

Gaze item type: <class 'h5py._hl.dataset.Dataset'>
Gaze item attributes:
  H5PATH: b'/#refs#/c'
  MATLAB_class: b'table'
  MATLAB_object_decode: 3

Gaze item dtype: uint32
No field names in dtype (not a structured array)

=== Checking if data is stored as MATLAB struct ===

Parent (gaze_refs_dataset) attributes:

=== Analyzing the 6-value structure ===
According to spec, gaze should have 21 columns:
Expected: 21 columns
Found: 6 values

The (1,6) structure might be:
  [large_num, 2, 1, 1, index, 1]
  Possibly: [address/ref, type, ?, ?, sequence, ?]
